# 08. Synthetic Data Recipe

- Parent issue: `#24`
- Status: `active`
- Summary: Implement synthetic data generation pipeline with solver-first / LLM-fallback teacher policy, quality filters, cost cap, smoke-run mode, and SHA-256 dataset fingerprinting. Produces SFTExample JSONL batches for SFT training.

## Audience and Why It Matters

Data generation owners, cost reviewers, and non-technical stakeholders evaluating risk and provenance.


## Decision / Hypothesis

Synthetic data should only be promoted when answers are verifiable, costs are bounded, and provenance is explicit.


## Environment and Reproduction

- Python: 3.11+
- Run from repo root: `jupyter notebook notebooks/08_synthetic_data_recipe.ipynb`
- Requires: `data/analysis/retry_candidates.jsonl` from Branch 1 (#22)
- Requires: `Verifier` implementation from Branch 2 (#23) for solver-first policy
- Cost warning: full generation run capped at $20; set `smoke=True` for initial inspection
- Companion registry entry: `docs/execution/NOTEBOOKS.md`

In [ ]:
from pathlib import Path
import platform

REPO_ROOT = Path.cwd()

print(f"Repo root: {REPO_ROOT}")
print(f"Python platform: {platform.platform()}")


In [ ]:
# Device detection: prefers CUDA, falls back to CPU gracefully.
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir() and (_candidate / 'notebooks').is_dir():
        REPO_ROOT = _candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.device import log_device_info
device = log_device_info()


## Method and Outputs

### Method

1. Load retry candidates from `data/analysis/retry_candidates.jsonl`.
2. Show `SyntheticConfig` and teacher policy (verifier-first → LLM fallback).
3. Show `QualityFilter` gates: dedupe, boxed, token length, provenance, category.
4. Run `--dry-run` to print token/cost estimate before committing to a full run.
5. Run smoke set (≤50 examples) — **HITL checkpoint**: human inspects output before authorizing full generation.
6. Verify fingerprint file written alongside output JSONL.

### Outputs produced

- `data/synthetic/<batch_id>.jsonl` — quality-filtered SFTExample rows
- `data/synthetic/<batch_id>.sha256` — SHA-256 fingerprint for reproducibility audit
- `configs/synthetic_prompts.yaml` — PromptCoT templates per category

In [ ]:
import sys
import json
from pathlib import Path

_cwd = Path.cwd()
for _candidate in [_cwd] + list(_cwd.parents):
    if (_candidate / 'src').is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from src.data.synthetic import (
    SyntheticConfig,
    QualityFilter,
    generate_from_retry_candidates,
    write_sft_examples_jsonl,
)
from src.evaluation.trajectory import TrajectoryRow

RETRY_CANDIDATES_PATH = Path("data/analysis/retry_candidates.jsonl")
OUTPUT_DIR = Path("data/synthetic")

print(f"retry_candidates path: {RETRY_CANDIDATES_PATH}")
print(f"output_dir: {OUTPUT_DIR}")

# Load candidates (or use empty list if not yet produced by Branch 1)
candidates: list[TrajectoryRow] = []
if RETRY_CANDIDATES_PATH.exists():
    # TrajectoryRow JSONL is not directly deserializable without the full
    # EvalRecord constructor; in the real run, use the trajectory module.
    lines = [l for l in RETRY_CANDIDATES_PATH.read_text().splitlines() if l.strip()]
    print(f"Loaded {len(lines)} retry candidate line(s) from {RETRY_CANDIDATES_PATH}")
else:
    print("WARNING: retry_candidates.jsonl not found — using empty list for scaffold demo.")

print(f"\nCandidates available: {len(candidates)}")

In [ ]:
# Show SyntheticConfig and teacher policy
config = SyntheticConfig(
    output_dir=OUTPUT_DIR,
    categories=["bit_manipulation", "math", "code", "science"],
    cost_cap_usd=20.0,
)

print("SyntheticConfig:")
print(f"  output_dir:    {config.output_dir}")
print(f"  categories:    {config.categories}")
print(f"  cost_cap_usd:  ${config.cost_cap_usd}")
print(f"  max_tokens:    {config.max_tokens}")

print("\nTeacher policy (in priority order):")
print("  1. Verifier.verify(pred, gold) — solver-first; uses Branch 2 Verifier Protocol")
print("  2. LLM teacher (DeepSeek-R1 → Claude → GPT-4 fallback)")
print("  answer=None from all → skip row")

print("\nQualityFilter gates:")
print("  - Dedupe: SHA-256 of example_id + category")
print("  - \\boxed{} present in completion")
print("  - Token length ≤ 8192")
print("  - Provenance: teacher, generated_at, source_run_id required")
print("  - Category in known taxonomy")

In [ ]:
# Dry-run: token/cost estimate
print("=== DRY RUN ===")
generate_from_retry_candidates(candidates, config, dry_run=True)
print("===============")
print()
print("Review the estimate above before authorizing a full run.")

In [ ]:
# Smoke run (≤50 examples)
# In a real run: set config.llm_teacher_fn to an actual API wrapper.
# Here we use a stub that returns a placeholder completion.
config.llm_teacher_fn = lambda prompt: rf"\boxed{{stub_answer}}"

print("Running smoke set (≤50 examples)...")
smoke_results = generate_from_retry_candidates(candidates, config, smoke=True)
print(f"Smoke results: {len(smoke_results)} SFTExample(s) passed quality filter")

# ── HITL CHECKPOINT ────────────────────────────────────────────────────────
print()
print("=" * 60)
print("HITL CHECKPOINT")
print("Inspect smoke_results above before authorizing full generation.")
print("Run generate_from_retry_candidates(candidates, config) without")
print("smoke=True only after human review confirms quality is acceptable.")
print("=" * 60)

In [ ]:
# Write batch and verify fingerprint (runs on smoke results)
import time
if smoke_results:
    batch_id = f"smoke_{int(time.time())}"
    out_path = OUTPUT_DIR / f"{batch_id}.jsonl"
    write_sft_examples_jsonl(smoke_results, out_path)
    sha_path = out_path.with_suffix(".sha256")
    print(f"Written:     {out_path}")
    print(f"Fingerprint: {sha_path}")
    print(f"SHA-256:     {sha_path.read_text().strip()}")
    print(f"Examples:    {len(smoke_results)}")
    for ex in smoke_results[:2]:
        print(f"  - {ex.example_id} | category={ex.category} | teacher={ex.provenance.get('teacher')}")
else:
    print("No smoke results to write (no candidates available).")
    print("Provide retry_candidates.jsonl from Branch 1 (#22) to generate examples.")

## Results / Open Risks

**Artifacts produced:**
- `src/data/synthetic.py` — generation pipeline with QualityFilter, teacher policy, cost cap
- `configs/synthetic_prompts.yaml` — PromptCoT templates per category
- `data/synthetic/<batch_id>.jsonl` + `.sha256` — generated SFTExample batches (after smoke approval)

**Open risks:**
- Requires `retry_candidates.jsonl` from Branch 1 (#22); scaffold runs with empty list.
- Real LLM teacher calls (`llm_teacher_fn`) must be configured before full generation — cost cap is enforced but API key setup is out of scope for this notebook.
- Smoke run HITL gate must be completed before authorizing full generation run.
- `Verifier.verify()` (Branch 2) must be instantiated and passed as `config.verifier` for solver-first policy.

## Sources

            - [Execution plan v0.2](docs/planning/plan_v0.2.md)
- [NVIDIA reasoning training blog](https://developer.nvidia.com/blog/train-a-reasoning-capable-llm-in-one-weekend-with-nvidia-nemo/)
- [SYNTHETIC-1](https://www.primeintellect.ai/blog/synthetic-1)
